In [ ]:
# 02_descriptive_analysis.ipynb
# 
# This notebook performs an exploratory analysis on the five
# raw datasets (before scaling) to characterize their
# distribution and structure. It includes:
#
#   1. General summary: number of instances, features, and
#      class distribution (defective vs. non-defective)
#   2. Feature means per dataset
#   3. Distribution plots of the target variable (countplot)
#   4. Histograms of key metrics: LOC and cyclomatic complexity v(g)
#   5. Boxplots of Halstead metrics: ev(g), iv(g), n, l, d, i
#   6. Correlation matrices between predictor variables
#
import sys
sys.path.append("../src")
from utils import DATASETS, TARGET

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

path_dataset = Path("../datasets")
results_path = Path("../results")
summary_path = results_path / "summary"
corr_path = results_path / "correlations"
plots_path = results_path / "plots"

summary_path.mkdir(parents=True, exist_ok=True)
corr_path.mkdir(parents=True, exist_ok=True)
plots_path.mkdir(parents=True, exist_ok=True)

data_dict = {
    ds: pd.read_csv(path_dataset / f"{ds}-T.csv").drop(
        columns=["project"], errors="ignore"
    )
    for ds in DATASETS
}

for ds in DATASETS:
    data_dict[ds][TARGET] = data_dict[ds][TARGET].astype(int)

summary = []

for ds, data in data_dict.items():
    n_instances = data.shape[0]
    n_features = data.drop(columns=[TARGET]).shape[1]

    defect_counts = data[TARGET].value_counts()
    defect_free = int(defect_counts.get(0, 0))
    defective = int(defect_counts.get(1, 0))

    defect_free_pct = defect_free / n_instances * 100
    defective_pct = defective / n_instances * 100

    summary.append({
        "Dataset": ds,
        "Instancias": n_instances,
        "Características": n_features,
        "Sin defectos": f"{defect_free} ({defect_free_pct:.2f}%)",
        "Con defectos": f"{defective} ({defective_pct:.2f}%)"
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(summary_path / "dataset_summary.csv", index=False)
print(summary_df.to_string(index=False))

summary_tables = []

for ds, data in data_dict.items():
    features = data.drop(columns=[TARGET])
    means = features.mean()
    table = pd.DataFrame({"Feature": means.index, ds: means.values})
    summary_tables.append(table)

final_table = summary_tables[0]
for t in summary_tables[1:]:
    final_table = final_table.merge(t, on="Feature")

final_table = final_table.round(2)
final_table.to_csv(summary_path / "feature_means_by_dataset.csv", index=False)

for ds, data in data_dict.items():
    print(f"\nDefect distribution - {ds}")
    print(data[TARGET].value_counts(normalize=True).round(4))

    fig, ax = plt.subplots()
    sns.countplot(x=TARGET, data=data, ax=ax)
    ax.set_title(f"Defect distribution - {ds}")
    ax.set_xlabel("Defect (0=No, 1=Yes)")
    plt.tight_layout()
    plt.savefig(plots_path / f"{ds}_defect_distribution.png", dpi=150)
    plt.close()

for ds, data in data_dict.items():
    if "loc" in data.columns:
        fig, ax = plt.subplots()
        sns.histplot(data=data, x="loc", bins=30, kde=True, ax=ax)
        ax.set_title(f"LOC distribution - {ds}")
        ax.set_xlabel("Lines of Code")
        plt.tight_layout()
        plt.savefig(plots_path / f"{ds}_loc_distribution.png", dpi=150)
        plt.close()

for ds, data in data_dict.items():
    if "v(g)" in data.columns:
        fig, ax = plt.subplots()
        sns.histplot(data=data, x="v(g)", bins=30, kde=True, ax=ax)
        ax.set_title(f"Cyclomatic Complexity v(g) distribution - {ds}")
        ax.set_xlabel("Cyclomatic Complexity")
        plt.tight_layout()
        plt.savefig(plots_path / f"{ds}_vg_distribution.png", dpi=150)
        plt.close()

metrics = ["ev(g)", "iv(g)", "n", "l", "d", "i"]

for ds, data in data_dict.items():
    for m in metrics:
        if m in data.columns:
            fig, ax = plt.subplots()
            sns.boxplot(x=data[m], ax=ax)
            ax.set_title(f"Distribution of {m} - {ds}")
            plt.tight_layout()
            plt.savefig(plots_path / f"{ds}_{m.replace('(', '').replace(')', '')}_boxplot.png", dpi=150)
            plt.close()

for ds, data in data_dict.items():
    corr = data.drop(columns=[TARGET], errors="ignore").corr()

    corr.to_csv(corr_path / f"{ds}_correlation_matrix.csv")

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"Correlation matrix - {ds}")
    plt.tight_layout()
    plt.savefig(corr_path / f"{ds}_correlation_matrix.png", dpi=150)
    plt.close()
